In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/inventory-optimization-ml-or

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

!git config --global user.email "smrititubid@gmail.com"
!git config --global user.name "smrititubid-afk"

Mounted at /content/drive
/content/drive/MyDrive/inventory-optimization-ml-or


In [2]:
df = pd.read_csv("data/processed/features_phase2.csv")

print(df.shape)
df.head()

(197925, 31)


,id,item_id,dept_id,cat_id,store_id,state_id,avg_demand,std_demand,cv,zero_demand_ratio,...,month_num,quarter,year_num,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_7,rolling_std_28
0,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,3,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,3,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
max_day = df['day'].max()

test_days = 28
calib_days = 28

train_df = df[df['day'] <= max_day - test_days - calib_days]

calib_df = df[
    (df['day'] > max_day - test_days - calib_days) &
    (df['day'] <= max_day - test_days)
]

test_df = df[df['day'] > max_day - test_days]

print("Train:", train_df.shape)
print("Calibration:", calib_df.shape)
print("Test:", test_df.shape)

Train: (192045, 31)
Calibration: (2940, 31)
Test: (2940, 31)


In [4]:
features = [
    'lag_1',
    'lag_7',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
    'rolling_std_7',
    'rolling_std_28',
    'day_of_week',
    'is_weekend',
    'month_num'
]

target = 'sales'

x_train = train_df[features]
y_train = train_df[target]

x_calib = calib_df[features]
y_calib = calib_df[target]

x_test = test_df[features]
y_test = test_df[target]

In [5]:
import lightgbm as lgb

model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

model.fit(x_train, y_train)

print("Model trained.")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.179997 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1046
[LightGBM] [Info] Number of data points in the train set: 192045, number of used features: 10
[LightGBM] [Info] Start training from score 0.680059
Model trained.


In [6]:
calib_pred = model.predict(x_calib)

calib_errors = np.abs(y_calib - calib_pred)

calib_errors[:10]

,sales
1829,0.224654
1830,0.220710
1831,0.220710
1832,0.099996
1833,0.120517
1834,0.146434
1835,0.146434
1836,0.103940
1837,0.099996
1838,0.099996


In [7]:
alpha = 0.10   # 90% prediction interval

q_hat = np.quantile(
    calib_errors,
    1 - alpha
)

print("Conformal error quantile:", q_hat)

Conformal error quantile: 1.7232685745382368


In [8]:
test_pred = model.predict(x_test)

lower_bound = test_pred - q_hat
upper_bound = test_pred + q_hat

lower_bound = np.maximum(lower_bound, 0)

results = test_df.copy()

results['prediction'] = test_pred
results['lower_bound'] = lower_bound
results['upper_bound'] = upper_bound

results[
    ['id', 'day', 'sales', 'prediction', 'lower_bound', 'upper_bound']
].head()

,id,day,sales,prediction,lower_bound,upper_bound
1857,FOODS_1_022_CA_4_validation,1886,0,0.087015,0.0,1.810284
1858,FOODS_1_022_CA_4_validation,1887,0,0.083071,0.0,1.806339
1859,FOODS_1_022_CA_4_validation,1888,0,0.083071,0.0,1.806339
1860,FOODS_1_022_CA_4_validation,1889,0,0.083071,0.0,1.806339
1861,FOODS_1_022_CA_4_validation,1890,0,0.103592,0.0,1.826860


In [9]:
covered = (
    (results['sales'] >= results['lower_bound']) &
    (results['sales'] <= results['upper_bound'])
)

coverage = covered.mean()

print("Empirical Coverage:", coverage)

Empirical Coverage: 0.8908163265306123


The conformal prediction framework achieved an empirical coverage of 89.1% against a target coverage of 90%, validating the effectiveness of distribution-free uncertainty quantification on intermittent demand data.

Baseline LightGBM forecasting generated point predictions but failed to quantify uncertainty associated with rare demand spikes. Conformal Prediction was used to calibrate forecast residuals and generate statistically valid prediction intervals. The resulting framework achieved 89.1% empirical coverage for a nominal 90% confidence level, providing uncertainty-aware demand estimates suitable for inventory decision making.

In [10]:
results.to_csv(
    "data/processed/conformal_predictions_phase3.csv",
    index=False
)

print("Conformal predictions saved!")

Conformal predictions saved!


In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/05_conformal_prediction.ipynb" \
"/content/drive/MyDrive/inventory-optimization-ml-or/notebooks/04_forecast_diagnostics.ipynb"